In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import warnings
import mlflow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlflow.tracking import MlflowClient

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="talk")

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_NAME = "Small_Data_Optimization"

In [ ]:
def infer_method(row: pd.Series, parent_method_lookup: dict[str, str]) -> str:
    if pd.notna(row.get("metrics.mask_flip_rate")):
        return "Evolutionary Algorithm"

    method = row.get("params.method")
    if pd.notna(method) and str(method).strip():
        method_text = str(method).strip().lower()
        if method_text == "grid_search":
            return "GridSearchCV"
        if method_text == "random_search":
            return "RandomizedSearchCV"
        if method_text == "optuna":
            return "Optuna"
        return str(method)

    parent_run_id = row.get("tags.mlflow.parentRunId")
    if pd.notna(parent_run_id):
        parent_method = parent_method_lookup.get(str(parent_run_id))
        if parent_method:
            return parent_method

    run_name = str(row.get("tags.mlflow.runName", ""))
    if "grid_search" in run_name:
        return "GridSearchCV"
    if "random_search" in run_name:
        return "RandomizedSearchCV"
    if "optuna" in run_name:
        return "Optuna"
    if "fold_" in run_name:
        return "Evolutionary Algorithm"

    return "Unknown"

client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f"Experiment '{EXPERIMENT_NAME}' was not found in MLflow.")

runs_df = mlflow.search_runs([experiment.experiment_id], output_format="pandas")

parent_method_lookup = {}
parent_dataset_lookup = {}
if not runs_df.empty:
    parent_rows = runs_df.loc[runs_df["tags.mlflow.parentRunId"].isna()].copy()
    for _, parent_row in parent_rows.iterrows():
        parent_run_id = str(parent_row["run_id"])
        parent_method = str(parent_row.get("params.method", ""))
        parent_dataset = str(parent_row.get("params.dataset", ""))

        if parent_method and parent_method.lower() != "nan":
            parent_method_lookup[parent_run_id] = infer_method(parent_row, {})
        elif pd.notna(parent_row.get("params.selection_operator")):
            parent_method_lookup[parent_run_id] = "Evolutionary Algorithm"

        if parent_dataset and parent_dataset.lower() != "nan":
            parent_dataset_lookup[parent_run_id] = parent_dataset

if runs_df.empty:
    fold_level_df = pd.DataFrame(columns=["dataset", "method", "outer_fold", "cv_mean", "cv_dispersion", "test_acc", "generalization_gap"])
    print(f"No runs were found for experiment '{EXPERIMENT_NAME}'. The notebook will still create empty summaries and skip plots.")
else:
    runs_df["method"] = runs_df.apply(lambda row: infer_method(row, parent_method_lookup), axis=1)
    runs_df["dataset"] = runs_df["params.dataset"]
    runs_df["dataset"] = runs_df["dataset"].fillna(runs_df["tags.mlflow.parentRunId"].map(parent_dataset_lookup))
    runs_df["dataset"] = runs_df["dataset"].fillna(runs_df["tags.mlflow.runName"].str.split("_fold_").str[0])
    runs_df["outer_fold"] = pd.to_numeric(runs_df.get("params.outer_fold"), errors="coerce")
    runs_df["outer_fold"] = runs_df["outer_fold"].fillna(
        pd.to_numeric(runs_df["tags.mlflow.runName"].str.extract(r"_fold_(\d+)")[0], errors="coerce")
    )

    fold_level_df = runs_df.loc[
        runs_df["metrics.cv_mean"].notna() & runs_df["metrics.test_acc"].notna()
    ].copy()

    fold_level_df["cv_mean"] = pd.to_numeric(fold_level_df["metrics.cv_mean"], errors="coerce")
    fold_level_df["cv_dispersion"] = pd.to_numeric(fold_level_df["metrics.cv_std"], errors="coerce")
    fold_level_df["test_acc"] = pd.to_numeric(fold_level_df["metrics.test_acc"], errors="coerce")
    fold_level_df["generalization_gap"] = fold_level_df["cv_mean"] - fold_level_df["test_acc"]

    fold_level_df = fold_level_df.loc[fold_level_df["method"] != "Unknown"].copy()

fold_level_df[["dataset", "method", "outer_fold", "cv_mean", "cv_dispersion", "test_acc", "generalization_gap"]].head()

In [ ]:
if fold_level_df.empty:
    summary_df = pd.DataFrame(columns=["dataset", "method", "n_runs", "avg_test_acc", "avg_cv_mean", "avg_cv_dispersion", "avg_generalization_gap"])
    method_summary_df = pd.DataFrame(columns=["method", "n_runs", "avg_test_acc", "avg_cv_mean", "avg_cv_dispersion", "avg_generalization_gap"])
else:
    summary_df = (
        fold_level_df
        .groupby(["dataset", "method"], as_index=False)
        .agg(
            n_runs=("test_acc", "size"),
            avg_test_acc=("test_acc", "mean"),
            avg_cv_mean=("cv_mean", "mean"),
            avg_cv_dispersion=("cv_dispersion", "mean"),
            avg_generalization_gap=("generalization_gap", "mean"),
        )
    )

    method_summary_df = (
        fold_level_df
        .groupby("method", as_index=False)
        .agg(
            n_runs=("test_acc", "size"),
            avg_test_acc=("test_acc", "mean"),
            avg_cv_mean=("cv_mean", "mean"),
            avg_cv_dispersion=("cv_dispersion", "mean"),
            avg_generalization_gap=("generalization_gap", "mean"),
        )
        .sort_values("avg_test_acc", ascending=False)
    )

summary_df.to_csv(ARTIFACT_DIR / "mlflow_method_summary.csv", index=False)
method_summary_df.to_csv(ARTIFACT_DIR / "mlflow_global_method_summary.csv", index=False)
summary_df.sort_values(["dataset", "method"])

In [ ]:
if fold_level_df.empty:
    best_runs_df = pd.DataFrame(columns=[
        "dataset",
        "method",
        "outer_fold",
        "cv_mean",
        "cv_dispersion",
        "test_acc",
        "generalization_gap",
        "learning_metrics",
    ])
else:
    candidate_columns = [
        "dataset",
        "method",
        "outer_fold",
        "cv_mean",
        "cv_dispersion",
        "test_acc",
        "generalization_gap",
        "params.C",
        "params.gamma",
        "params.kernel",
        "metrics.selected_feature_count",
        "metrics.sigma_c",
        "metrics.sigma_gamma",
        "metrics.mask_flip_rate",
    ]
    best_run_source_df = runs_df.loc[runs_df["metrics.test_acc"].notna()].copy()
    best_run_source_df["method"] = best_run_source_df.apply(lambda row: infer_method(row, parent_method_lookup), axis=1)
    best_run_source_df["dataset"] = best_run_source_df["params.dataset"]
    best_run_source_df["dataset"] = best_run_source_df["dataset"].fillna(best_run_source_df["tags.mlflow.parentRunId"].map(parent_dataset_lookup))
    best_run_source_df["dataset"] = best_run_source_df["dataset"].fillna(best_run_source_df["tags.mlflow.runName"].str.split("_fold_").str[0])
    best_run_source_df["outer_fold"] = pd.to_numeric(best_run_source_df["params.outer_fold"], errors="coerce")
    best_run_source_df["outer_fold"] = best_run_source_df["outer_fold"].fillna(
        pd.to_numeric(best_run_source_df["tags.mlflow.runName"].str.extract(r"_fold_(\d+)")[0], errors="coerce")
    )
    best_run_source_df["cv_mean"] = pd.to_numeric(best_run_source_df["metrics.cv_mean"], errors="coerce")
    best_run_source_df["cv_dispersion"] = pd.to_numeric(best_run_source_df["metrics.cv_std"], errors="coerce")
    best_run_source_df["test_acc"] = pd.to_numeric(best_run_source_df["metrics.test_acc"], errors="coerce")
    best_run_source_df["generalization_gap"] = best_run_source_df["cv_mean"] - best_run_source_df["test_acc"]

    best_runs_df = (
        best_run_source_df
        .sort_values(["dataset", "method", "test_acc", "cv_mean", "generalization_gap"], ascending=[True, True, False, False, True])
        .groupby(["dataset", "method"], as_index=False)
        .head(1)
        .loc[:, candidate_columns + ["tags.mlflow.runName"]]
        .copy()
    )

    best_runs_df = best_runs_df.rename(columns={
        "params.C": "C",
        "params.gamma": "gamma",
        "params.kernel": "kernel",
        "metrics.selected_feature_count": "selected_feature_count",
        "metrics.sigma_c": "sigma_c",
        "metrics.sigma_gamma": "sigma_gamma",
        "metrics.mask_flip_rate": "mask_flip_rate",
        "tags.mlflow.runName": "run_name",
    })

    best_runs_df["learning_metrics"] = best_runs_df.apply(
        lambda row: json.dumps({
            "cv_mean": None if pd.isna(row["cv_mean"]) else round(float(row["cv_mean"]), 6),
            "cv_dispersion": None if pd.isna(row["cv_dispersion"]) else round(float(row["cv_dispersion"]), 6),
            "test_acc": None if pd.isna(row["test_acc"]) else round(float(row["test_acc"]), 6),
            "generalization_gap": None if pd.isna(row["generalization_gap"]) else round(float(row["generalization_gap"]), 6),
            "selected_feature_count": None if pd.isna(row["selected_feature_count"]) else int(row["selected_feature_count"]),
            "sigma_c": None if pd.isna(row["sigma_c"]) else round(float(row["sigma_c"]), 6),
            "sigma_gamma": None if pd.isna(row["sigma_gamma"]) else round(float(row["sigma_gamma"]), 6),
            "mask_flip_rate": None if pd.isna(row["mask_flip_rate"]) else round(float(row["mask_flip_rate"]), 6),
        }),
        axis=1,
    )

    best_runs_df.to_csv(ARTIFACT_DIR / "mlflow_best_runs_by_dataset_and_method.csv", index=False)

best_runs_df

In [ ]:
order = ["GridSearchCV", "RandomizedSearchCV", "Optuna", "Evolutionary Algorithm"]
learning_curve_path = ARTIFACT_DIR / "learning_metrics_by_method_and_dataset.png"

if fold_level_df.empty:
    print("Skipping learning graph because no fold-level runs are available yet.")
else:
    learning_long_df = fold_level_df.loc[fold_level_df["method"].isin(order)].copy()
    learning_long_df = learning_long_df.dropna(subset=["outer_fold"])
    learning_long_df["outer_fold"] = learning_long_df["outer_fold"].astype(int)
    learning_long_df = learning_long_df.melt(
        id_vars=["dataset", "method", "outer_fold"],
        value_vars=["cv_mean", "test_acc", "cv_dispersion"],
        var_name="metric",
        value_name="score",
    )
    learning_long_df["metric"] = learning_long_df["metric"].map({
        "cv_mean": "CV Mean",
        "test_acc": "Test Acc",
        "cv_dispersion": "CV Dispersion",
    })

    g = sns.relplot(
        data=learning_long_df,
        kind="line",
        x="outer_fold",
        y="score",
        hue="method",
        marker="o",
        col="dataset",
        row="metric",
        height=3.2,
        aspect=1.15,
        facet_kws={"sharex": True, "sharey": False},
    )
    g.set_titles("{row_name} | {col_name}")
    g.set_axis_labels("Outer Fold", "Score")
    for ax in g.axes.flat:
        ax.grid(True, alpha=0.25)
    g._legend.set_title("Method")
    plt.subplots_adjust(top=0.92)
    g.fig.suptitle("Learning Graph: Metric Evolution Across Outer Folds")
    g.fig.savefig(learning_curve_path, dpi=180, bbox_inches="tight")
    plt.show()

learning_curve_path

In [ ]:
order = ["GridSearchCV", "RandomizedSearchCV", "Optuna", "Evolutionary Algorithm"]
boxplot_path = ARTIFACT_DIR / "test_acc_dispersion_boxplot.png"

if fold_level_df.empty:
    print("Skipping boxplot because no fold-level runs are available yet.")
else:
    plt.figure(figsize=(12, 6))
    plot_df = fold_level_df.loc[fold_level_df["method"].isin(order)].copy()
    sns.boxplot(data=plot_df, x="method", y="test_acc", order=order, palette="Set2")
    plt.title("Fold-Level Test Accuracy Dispersion by Method")
    plt.xlabel("Method")
    plt.ylabel("Test Accuracy")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(boxplot_path, dpi=180, bbox_inches="tight")
    plt.show()

boxplot_path

In [ ]:
barplot_path = ARTIFACT_DIR / "generalization_gap_barplot.png"

if method_summary_df.empty:
    print("Skipping bar chart because no method summaries are available yet.")
else:
    plt.figure(figsize=(13, 6))
    method_summary_plot = method_summary_df.loc[method_summary_df["method"].isin(order)].copy()
    method_summary_plot = method_summary_plot.melt(
        id_vars=["method"],
        value_vars=["avg_cv_mean", "avg_test_acc"],
        var_name="metric",
        value_name="score",
    )
    metric_labels = {"avg_cv_mean": "CV Mean", "avg_test_acc": "Test Acc"}
    method_summary_plot["metric"] = method_summary_plot["metric"].map(metric_labels)
    sns.barplot(data=method_summary_plot, x="method", y="score", hue="metric", order=order, palette="Dark2")
    plt.title("Generalization Gap: CV Mean vs Test Accuracy")
    plt.xlabel("Method")
    plt.ylabel("Score")
    plt.xticks(rotation=20, ha="right")
    plt.legend(title="Metric")
    plt.tight_layout()
    plt.savefig(barplot_path, dpi=180, bbox_inches="tight")
    plt.show()

barplot_path

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlflow.tracking import MlflowClient

sns.set_theme(style="whitegrid", context="talk")

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

client = MlflowClient()
experiment = client.get_experiment_by_name("Small_Data_Optimization")

if experiment is None:
    print("Експеримент не знайдено.")
else:
    runs = client.search_runs(experiment_ids=[experiment.experiment_id])
    
    datasets = ['sonar', 'ionosphere', 'wine', 'stress_classification']
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    for i, dataset in enumerate(datasets):
        fold_runs = [r for r in runs if r.data.tags.get("mlflow.runName", "") == f"{dataset}_fold_1"]
        
        if not fold_runs:
            axes[i].set_title(f"Немає даних для {dataset.upper()}")
            continue
            
        run_id = fold_runs[0].info.run_id
        
        try:
            local_path = client.download_artifacts(run_id, "generation_history.csv")
            history_df = pd.read_csv(local_path)
            
            axes[i].plot(history_df['generation'], history_df['best_mean_acc'], 
                         label='CV Точність ($\mu$)', color='#2ca02c', linewidth=2.5)
            axes[i].plot(history_df['generation'], history_df['best_fitness'], 
                         label='Фітнес ($\mu - 2\sigma$)', color='#1f77b4', linestyle='--', linewidth=2.5)

            axes[i].plot(history_df['generation'], history_df['population_mean_fitness'], 
                         label='Середнє популяції', color='#ff7f0e', alpha=0.5, linewidth=1.5)
            
            axes[i].set_title(f"Збіжність ЕА: {dataset.upper()}", fontsize=16, fontweight='bold')
            axes[i].set_xlabel("Покоління (Епохи)", fontsize=14)
            axes[i].set_ylabel("Оцінка", fontsize=14)
            axes[i].legend(fontsize=12)
            
        except Exception as e:
            axes[i].set_title(f"Помилка завантаження {dataset.upper()}")
            print(f"Не вдалося завантажити артефакт для {dataset}: {e}")

    plt.tight_layout()
    
    # Зберігаємо графік у папку artifacts
    plot_path = ARTIFACT_DIR / "ea_convergence_epochs.png"
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    print(f"Графік збіжності збережено: {plot_path.resolve()}")
    plt.show()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlflow.tracking import MlflowClient
from pathlib import Path

sns.set_theme(style="whitegrid", context="talk")
ARTIFACT_DIR = Path("artifacts")

client = MlflowClient()
experiment = client.get_experiment_by_name("Small_Data_Optimization")

if experiment is None:
    print("Експеримент не знайдено.")
else:
    runs = client.search_runs(experiment_ids=[experiment.experiment_id])

    datasets = ['sonar', 'ionosphere', 'wine', 'stress_classification'] 
    fig, axes = plt.subplots(1, len(datasets), figsize=(7 * len(datasets), 6))

    if len(datasets) == 1: axes = [axes]
    
    for i, dataset in enumerate(datasets):
        all_folds_history = []

        for fold in range(1, 6):
            fold_runs = [r for r in runs if r.data.tags.get("mlflow.runName", "") == f"{dataset}_fold_{fold}"]
            if not fold_runs: 
                continue
            
            run_id = fold_runs[0].info.run_id
            try:
                local_path = client.download_artifacts(run_id, "generation_history.csv")
                df = pd.read_csv(local_path)
                df['fold'] = fold
                all_folds_history.append(df)
            except Exception:
                pass
            
        if not all_folds_history:
            axes[i].set_title(f"Немає даних для {dataset.upper()}")
            continue
            
        combined_df = pd.concat(all_folds_history, ignore_index=True)
        
        sns.lineplot(data=combined_df, x='generation', y='best_mean_acc', 
                     ax=axes[i], label='CV Точність ($\mu$)', color='#2ca02c', errorbar='sd')
        sns.lineplot(data=combined_df, x='generation', y='best_fitness', 
                     ax=axes[i], label='Фітнес ($\mu - 2\sigma$)', color='#1f77b4', linestyle='--', errorbar='sd')
        sns.lineplot(data=combined_df, x='generation', y='population_mean_fitness', 
                     ax=axes[i], label='Середнє популяції', color='#ff7f0e', alpha=0.5, errorbar=None)
        
        axes[i].set_title(f"Усереднена збіжність ЕА: {dataset.upper()}", fontsize=16, fontweight='bold')
        axes[i].set_xlabel("Покоління (Епохи)", fontsize=14)
        axes[i].set_ylabel("Оцінка", fontsize=14)
        axes[i].legend(fontsize=12)

    plt.tight_layout()
    plot_path = ARTIFACT_DIR / "ea_convergence_all_folds.png"
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    print(f"Графік усередненої збіжності збережено: {plot_path.resolve()}")
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlflow.tracking import MlflowClient
from pathlib import Path

sns.set_theme(style="whitegrid", context="talk")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

client = MlflowClient()
experiment = client.get_experiment_by_name("Small_Data_Optimization")

if experiment is None:
    print("Експеримент не знайдено.")
else:
    runs = client.search_runs(experiment_ids=[experiment.experiment_id])
    datasets = ['sonar', 'ionosphere', 'wine', 'stress_classification']
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    for i, dataset in enumerate(datasets):
        ax = axes[i]
        optuna_histories = []
        ea_histories = []

        for fold in range(1, 6):
            opt_runs = [r for r in runs if r.data.tags.get("mlflow.runName", "") == f"{dataset}_optuna_fold_{fold}"]
            if opt_runs:
                try:
                    run_id = opt_runs[0].info.run_id
                    opt_path = client.download_artifacts(run_id, f"optimization_history/{dataset}_optuna_fold_{fold}_history.csv")
                    df_opt = pd.read_csv(opt_path)
                    df_opt['evaluation_budget'] = df_opt['trial']  
                    optuna_histories.append(df_opt)
                except Exception:
                    pass
            
            ea_runs = [r for r in runs if r.data.tags.get("mlflow.runName", "") == f"{dataset}_fold_{fold}"]
            if ea_runs:
                try:
                    run_id = ea_runs[0].info.run_id
                    ea_path = client.download_artifacts(run_id, "generation_history.csv")
                    df_ea = pd.read_csv(ea_path)
                    df_ea['evaluation_budget'] = df_ea['generation'] * 20  
                    ea_histories.append(df_ea)
                except Exception:
                    pass

        if optuna_histories:
            combined_opt = pd.concat(optuna_histories, ignore_index=True)
            sns.lineplot(data=combined_opt, x='evaluation_budget', y='best_value',
                         ax=ax, label='Optuna (Bayesian)', color='#d62728', errorbar='sd', linewidth=2)
        
        if ea_histories:
            combined_ea = pd.concat(ea_histories, ignore_index=True)
            sns.lineplot(data=combined_ea, x='evaluation_budget', y='best_mean_acc',
                         ax=ax, label='EA (Наш метод)', color='#1f77b4', errorbar='sd', linewidth=2)
                         
        ax.set_title(f"Датасет: {dataset.upper()}", fontweight='bold', fontsize=16)
        ax.set_xlabel("Бюджет оцінок", fontsize=14)
        ax.set_ylabel("Найкраще значення CV Mean", fontsize=14)
        ax.legend(fontsize=12)

    plt.tight_layout()
    plot_path = ARTIFACT_DIR / "convergence_comparison_all_datasets.png"
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    print(f"Графік успішно збережено: {plot_path.resolve()}")
    plt.show()